# CLM-8B (Contrastive-LM) on a Colab L4 GPU
**Runtime → Change runtime type → L4 GPU** (Colab Pro or pay-as-you-go compute units). A free T4 is too small for the Qwen3-8B encoder in bf16 (~16.4 GB).
Needs `MyDrive/fin-decider/fin-decider-colab.tar.gz` (built on the Mac with `colab/package.sh`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'{gb:.1f} GB'); assert gb > 20, 'switch the runtime to an L4 (24 GB) or A100'

In [ ]:
# 1. Mount Drive, unpack the package, check out the exact commit, install the pipeline.
from google.colab import drive
drive.mount('/content/drive')
import json, os, subprocess, time
DRIVE = '/content/drive/MyDrive/fin-decider'
OUT = f'{DRIVE}/results'
os.makedirs(OUT, exist_ok=True)
!rm -rf /content/pkg && mkdir -p /content/pkg && tar -xzf {DRIVE}/fin-decider-colab.tar.gz -C /content/pkg
BUNDLE = json.load(open('/content/pkg/bundle.json')); print(BUNDLE)
!rm -rf /content/fin-decider && git clone -q /content/pkg/repo.bundle /content/fin-decider -b {BUNDLE['branch']}
%cd /content/fin-decider
!git checkout -q {BUNDLE['commit']} && git log --oneline -1
!cp -r /content/pkg/corpus_native . && (test -d /content/pkg/corpus && cp -r /content/pkg/corpus . || echo "no Bloomberg corpus in package (Reuters only)")

MODEL = 'clm'

In [ ]:
# 2. Model packages.
!pip install -q vllm==0.32.0 contrastive-lm==0.1.0

In [ ]:
# 3. Install the pipeline after the model package, so pip keeps the model's pinned dependencies.
!pip install -q -e . && classify --help | head -3
!pip freeze > {OUT}/pip_freeze_{MODEL}.txt; nvidia-smi > {OUT}/gpu_{MODEL}.txt

In [ ]:
# 4. Start the Qwen3-8B encoder (vLLM, pinned revision) and the CLM server; wait for both.
import requests
def wait(url, proc, name, tries=120):
    for _ in range(tries):
        try:
            if requests.get(url, timeout=10).ok:
                print(name, 'ready'); return
        except requests.RequestException:
            pass
        if proc.poll() is not None:
            raise RuntimeError(f'{name} exited; see its log on Drive')
        time.sleep(10)
    raise TimeoutError(name)
vlog = open(f'{OUT}/clm_vllm.log', 'a')
encoder = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3-8B', '--revision', 'b968826d9c46dd6066d109eabc6255188de91218',
                            '--served-model-name', 'qwen3-8b', '--runner', 'pooling', '--max-model-len', '2048',
                            '--dtype', 'bfloat16', '--port', '8090'], stdout=vlog, stderr=subprocess.STDOUT)
wait('http://127.0.0.1:8090/health', encoder, 'encoder', tries=180)   # first start downloads ~16 GB
clog = open(f'{OUT}/clm_server.log', 'a')
server = subprocess.Popen(['clm-serve'], stdout=clog, stderr=subprocess.STDOUT)
wait('http://127.0.0.1:8700/health', server, 'clm-serve')
probe = {'state': 'Bank of America arranged a 500 million dollar syndicated loan for a utility.',
         'questions': {'q1': {'type': 'noul', 'instructions': 'Does the article describe a bank lending money?'}}}
r = requests.post('http://127.0.0.1:8700/v1/systemone', json=probe, timeout=120)
print(r.status_code); print(json.dumps(r.json(), indent=1)[:1500])
print(json.dumps(requests.get('http://127.0.0.1:8700/health').json(), indent=1)[:800])
MODEL_NAME = r.json().get('model')
json.dump(r.json(), open(f'{OUT}/clm_probe.json', 'w'), indent=1)
print('model name reported by the server:', MODEL_NAME)
over = (f'--set models.clm.expected_model_returned={MODEL_NAME}' if MODEL_NAME
        else '--set models.clm.allow_returned_model_mismatch=true')
print(over)

## Run clm on the Reuters top-10 test set (2,533 stories, v1 questions)
Comparable with the Jev, d1 and Clef v1 runs. Results and logs are written to Drive, and `--resume` continues after a disconnect: just rerun the setup cells, the server cell and this cell.

In [ ]:
!classify run -c configs/native_reuters_top10_local.yaml -m clm --no-dev-eval --resume \
   --output-dir {OUT}/classification_results_native --reports-dir {OUT}/reports {over} 2>&1 | tee -a {OUT}/clm_reuters.log

## Optional: clm on the first 1,000 Bloomberg articles (G-SIB ontology)
The same 1,000 articles as the Jev, d1, Clef and Laya engineering runs; includes the synthetic dev-set check. Needs the full package (not `--reuters-only`).

In [ ]:
!classify run -c configs/classification.yaml -m clm --limit 1000 --resume --set models.clm.enabled=true \
   --output-dir {OUT}/classification_results --reports-dir {OUT}/reports {over} 2>&1 | tee -a {OUT}/clm_bloomberg1000.log

## Done
Download `MyDrive/fin-decider/results/` and tell Claude where it is; the runs are merged into the repo's results folders and scored there.